In [1]:
import os
import pandas as pd
import boto3
from sagemaker import get_execution_role
from pprint import pprint
import json
import time

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_step = os.getcwd().split('/')[-1]
print(f'Step: {str_step}')

# name function
str_name = 'step-genxii-ad-model-boto3'

Project: 20231010-gen-xii
Step: 12_step_function


### Hyperparameter df

In [3]:
# make a dictionary of hyperparameters, save as df to s3, so I can re-convert it to dict in the images
dict_hyperparameters = {
    # data sets
    'STR_FILENAME_TRAIN': 'df_train_noleaks_pre.gzip',
    'STR_FILENAME_VALID': 'df_valid_noleaks_pre.gzip', # always use the full data set for the valid model
    'STR_FILENAME_TEST': 'df_test_noleaks_pre.gzip', # always use the full data set for the test model
    # ITERATIONS - define once for consistency
    'INT_N_ITERATIONS': 1000,
    # proportion of iterations used for early stopping
    'PROP_EARLY_STOPPING': 0.05,
    # tuning - 2
    'INT_N_TUNING_JOBS_2': 50, # number of tuning jobs in the second tuning job
    # eval metric
    'STR_EVAL_METRIC': 'AUC',
}

# make df
df = pd.DataFrame(dict_hyperparameters.items(), columns=['keys','values'])

# save
str_filename = 'df_hyperparameters.csv'
str_uri = f's3://{str_project}/01_ad/02_model/02_model/12_step_function/{str_filename}'
df.to_csv(str_uri, index=False)

# show
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:275: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,keys,values
0,STR_FILENAME_TRAIN,df_train_noleaks_pre.gzip
1,STR_FILENAME_VALID,df_valid_noleaks_pre.gzip
2,STR_FILENAME_TEST,df_test_noleaks_pre.gzip
3,INT_N_ITERATIONS,1000
4,PROP_EARLY_STOPPING,0.05
5,INT_N_TUNING_JOBS_2,50
6,STR_EVAL_METRIC,AUC


### Write ```definition.json```

In [4]:
%%writefile definition.json

{
  "Comment": "A description of my state machine",
  "StartAt": "GetStartingFeats2",
  "States": {
    "GetStartingFeats2": {
      "Type": "Task",
      "Resource": "arn:aws:states:::lambda:invoke",
      "Parameters": {
        "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxii-ad-starting-feats-2:$LATEST"
      },
      "Retry": [
        {
          "ErrorEquals": [
            "Lambda.ServiceException",
            "Lambda.AWSLambdaException",
            "Lambda.SdkClientException",
            "Lambda.TooManyRequestsException"
          ],
          "IntervalSeconds": 2,
          "MaxAttempts": 6,
          "BackoffRate": 2
        }
      ],
      "Next": "Tuning2"
    },
    "Tuning2": {
      "Type": "Task",
      "Resource": "arn:aws:states:::batch:submitJob.sync",
      "Parameters": {
        "JobName": "tuning-2",
        "JobDefinition": "arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxii-ad-tuning-2-1:7",
        "JobQueue": "arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxii-ad-tuning-2-1",
        "ArrayProperties": {
          "Size": "INT_N_TUNING_JOBS_2"
        }
      },
      "Next": "ConcatTuning2",
      "Retry": [
        {
          "ErrorEquals": [
            "States.ALL"
          ],
          "BackoffRate": 2,
          "IntervalSeconds": 1,
          "MaxAttempts": 5
        }
      ]
    },
    "ConcatTuning2": {
      "Type": "Task",
      "Resource": "arn:aws:states:::lambda:invoke",
      "Parameters": {
        "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxii-ad-concat-tuning-2:$LATEST"
      },
      "Retry": [
        {
          "ErrorEquals": [
            "Lambda.ServiceException",
            "Lambda.AWSLambdaException",
            "Lambda.SdkClientException",
            "Lambda.TooManyRequestsException"
          ],
          "IntervalSeconds": 2,
          "MaxAttempts": 6,
          "BackoffRate": 2
        }
      ],
      "Next": "SensitivityTuning2"
    },
    "SensitivityTuning2": {
      "Type": "Map",
      "ItemProcessor": {
        "ProcessorConfig": {
          "Mode": "DISTRIBUTED",
          "ExecutionType": "STANDARD"
        },
        "StartAt": "SensitivityAnalysis2",
        "States": {
          "SensitivityAnalysis2": {
            "Type": "Task",
            "Resource": "arn:aws:states:::batch:submitJob.sync",
            "Parameters": {
              "Parameters": {
                "ColumnName.$": "$"
              },
              "JobName": "sensitivity-analysis",
              "JobDefinition": "arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxii-ad-sensitivity-1:7",
              "JobQueue": "arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxii-ad-sensitivity-1"
            },
            "End": true,
            "Retry": [
              {
                "ErrorEquals": [
                  "States.ALL"
                ],
                "BackoffRate": 2,
                "IntervalSeconds": 1,
                "MaxAttempts": 5
              }
            ]
          }
        }
      },
      "Label": "Map",
      "MaxConcurrency": 2000,
      "ItemReader": {
        "Resource": "arn:aws:states:::s3:getObject",
        "ReaderConfig": {
          "InputType": "JSON"
        },
        "Parameters": {
          "Bucket": "20231010-gen-xii",
          "Key": "01_ad/02_model/02_model/01_lambda_get_starting_feats/json_cols_in_model.json"
        }
      },
      "Next": "ConcatSensitivity2",
      "ResultWriter": {
        "Resource": "arn:aws:states:::s3:putObject",
        "Parameters": {
          "Bucket": "20231010-gen-xii",
          "Prefix": "01_ad/02_model/02_model/04_batch_sensitivity_analysis/iterations/"
        }
      }
    },
    "ConcatSensitivity2": {
      "Type": "Task",
      "Resource": "arn:aws:states:::lambda:invoke",
      "Parameters": {
        "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxii-ad-concat-sensitivity:$LATEST"
      },
      "Retry": [
        {
          "ErrorEquals": [
            "Lambda.ServiceException",
            "Lambda.AWSLambdaException",
            "Lambda.SdkClientException",
            "Lambda.TooManyRequestsException"
          ],
          "IntervalSeconds": 2,
          "MaxAttempts": 6,
          "BackoffRate": 2
        }
      ],
      "Next": "GetNFeats2"
    },
    "GetNFeats2": {
      "Type": "Task",
      "Resource": "arn:aws:states:::lambda:invoke",
      "OutputPath": "$.Payload",
      "Parameters": {
        "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxii-ad-get-n-feats:$LATEST"
      },
      "Retry": [
        {
          "ErrorEquals": [
            "Lambda.ServiceException",
            "Lambda.AWSLambdaException",
            "Lambda.SdkClientException",
            "Lambda.TooManyRequestsException"
          ],
          "IntervalSeconds": 2,
          "MaxAttempts": 6,
          "BackoffRate": 2
        }
      ],
      "Next": "Choice"
    },
    "Choice": {
      "Type": "Choice",
      "Choices": [
        {
          "Not": {
            "Variable": "$",
            "StringMatches": "0"
          },
          "Next": "UpdateFeatDropList2"
        }
      ],
      "Default": "SelectBestModel"
    },
    "SelectBestModel": {
      "Type": "Task",
      "Resource": "arn:aws:states:::lambda:invoke",
      "Parameters": {
        "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxii-ad-select-best-model:$LATEST"
      },
      "Retry": [
        {
          "ErrorEquals": [
            "Lambda.ServiceException",
            "Lambda.AWSLambdaException",
            "Lambda.SdkClientException",
            "Lambda.TooManyRequestsException"
          ],
          "IntervalSeconds": 1,
          "MaxAttempts": 3,
          "BackoffRate": 2
        }
      ],
      "Next": "ParallelValid"
    },
    "ParallelValid": {
      "Type": "Parallel",
      "Branches": [
        {
          "StartAt": "PDPlotsValid2",
          "States": {
            "PDPlotsValid2": {
              "Type": "Task",
              "Resource": "arn:aws:states:::batch:submitJob.sync",
              "Parameters": {
                "JobName": "pd-plots-valid",
                "JobDefinition": "arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxii-ad-plots-valid-1:9",
                "JobQueue": "arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxii-ad-plots-valid-1"
              },
              "End": true
            }
          }
        },
        {
          "StartAt": "ModelEvalValid2",
          "States": {
            "ModelEvalValid2": {
              "Type": "Task",
              "Resource": "arn:aws:states:::batch:submitJob.sync",
              "Parameters": {
                "JobName": "model-eval-valid",
                "JobDefinition": "arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxii-ad-eval-valid-1:7",
                "JobQueue": "arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxii-ad-eval-valid-1"
              },
              "End": true
            }
          }
        },
        {
          "StartAt": "Disparate2",
          "States": {
            "Disparate2": {
              "Type": "Task",
              "Resource": "arn:aws:states:::batch:submitJob.sync",
              "Parameters": {
                "JobName": "disparate",
                "JobDefinition": "arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxii-ad-disparate-valid-1:9",
                "JobQueue": "arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxii-ad-disparate-valid-1"
              },
              "End": true
            }
          }
        },
        {
          "StartAt": "PDPlotsTest2",
          "States": {
            "PDPlotsTest2": {
              "Type": "Task",
              "Resource": "arn:aws:states:::batch:submitJob.sync",
              "Parameters": {
                "JobName": "pd-plots-test",
                "JobDefinition": "arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxii-ad-plots-test-1:9",
                "JobQueue": "arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxii-ad-plots-test-1"
              },
              "End": true
            }
          }
        },
        {
          "StartAt": "ModelEvalTest2",
          "States": {
            "ModelEvalTest2": {
              "Type": "Task",
              "Resource": "arn:aws:states:::batch:submitJob.sync",
              "Parameters": {
                "JobName": "model-eval-test",
                "JobDefinition": "arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxii-ad-eval-test-1:7",
                "JobQueue": "arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxii-ad-eval-test-1"
              },
              "End": true
            }
          }
        }
      ],
      "End": true
    },
    "UpdateFeatDropList2": {
      "Type": "Task",
      "Resource": "arn:aws:states:::lambda:invoke",
      "Parameters": {
        "Payload.$": "$",
        "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxii-ad-update-feats:$LATEST"
      },
      "Retry": [
        {
          "ErrorEquals": [
            "Lambda.ServiceException",
            "Lambda.AWSLambdaException",
            "Lambda.SdkClientException",
            "Lambda.TooManyRequestsException"
          ],
          "IntervalSeconds": 2,
          "MaxAttempts": 6,
          "BackoffRate": 2
        }
      ],
      "Next": "GetStartingFeats2"
    }
  }
}

Writing definition.json


#### Replace string placeholders

In [5]:
# load it
dict_definition = json.load(open('./definition.json'))
# make into string
str_definition = json.dumps(dict_definition)

In [6]:
# INT_N_TUNING_JOBS_2
str_definition = str_definition.replace('"INT_N_TUNING_JOBS_2"', str(dict_hyperparameters['INT_N_TUNING_JOBS_2']))

### Create state machine

In [7]:
cls_client_sfn = boto3.client('stepfunctions')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# list state machines
dict_response = cls_client_sfn.list_state_machines(
)
list_dict_state_machines = dict_response['stateMachines']
list_dict_state_names = [{dict_state_machine['name']: dict_state_machine['stateMachineArn']} for dict_state_machine in list_dict_state_machines]
dict_state_names = {key: val for dict_name in list_dict_state_names for key, val in dict_name.items()}
pprint(dict_state_names)

{'MyStateMachine-fldl4s6of': 'arn:aws:states:us-west-2:836690756591:stateMachine:MyStateMachine-fldl4s6of',
 'gen-xi-retro-scoring': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xi-retro-scoring',
 'gen-xii-payload-parsing-jq': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-payload-parsing-jq',
 'gen-xii-retro-scoring': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-retro-scoring',
 'genxi-payload-parsing': 'arn:aws:states:us-west-2:836690756591:stateMachine:genxi-payload-parsing',
 'genxii-payload-parsing': 'arn:aws:states:us-west-2:836690756591:stateMachine:genxii-payload-parsing',
 'poc-step-genxii-lgd-lambda-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:poc-step-genxii-lgd-lambda-boto3',
 'poc-step-genxii-pd-lambda-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:poc-step-genxii-pd-lambda-boto3',
 'step-genxii-ad-feat-select-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-feat-select-boto3',
 '

In [10]:
# get list of just names
list_str_names = [list(dict_state_names.keys())[0] for dict_state_names in list_dict_state_names]
# if our name is in there
if str_name in list_str_names:
    print(f'State machine {str_name} exists, it will be deleted')
    str_arn = dict_state_names[str_name]
    print(f'Deleting {str_arn}')
    print('')
    dict_response = cls_client_sfn.delete_state_machine(
        stateMachineArn=str_arn,
    )
    pprint(dict_response)
else:
    print(f'State machine {str_name} does not exist, so it will not be deleted')

State machine step-genxii-ad-model-boto3 exists, it will be deleted
Deleting arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-model-boto3

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '2',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Mon, 06 May 2024 17:31:51 GMT',
                                      'x-amzn-requestid': 'd3ff0860-e481-4093-9cca-7f4c217a34f8'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'd3ff0860-e481-4093-9cca-7f4c217a34f8',
                      'RetryAttempts': 0}}


In [11]:
# make a state machine
while True:
    try:
        dict_response = cls_client_sfn.create_state_machine(
            name=str_name,
            definition=str_definition,
            roleArn=str_role,
            type='STANDARD',
        )
        pprint(dict_response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '131',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Mon, 06 May 2024 17:32:40 GMT',
                                      'x-amzn-requestid': '77d92f57-c6ae-4c16-86d1-a32bcb578a5b'},
                      'HTTPStatusCode': 200,
                      'RequestId': '77d92f57-c6ae-4c16-86d1-a32bcb578a5b',
                      'RetryAttempts': 0},
 'creationDate': datetime.datetime(2024, 5, 6, 17, 32, 40, 11000, tzinfo=tzlocal()),
 'stateMachineArn': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-model-boto3'}


### Describe state machine

In [12]:
str_state_machine_arn = dict_response['stateMachineArn']
print(f'State Machine ARN: {str_state_machine_arn}')
dict_response = cls_client_sfn.describe_state_machine(
    stateMachineArn=str_state_machine_arn,
)
pprint(dict_response)

State Machine ARN: arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-model-boto3
{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '7615',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Mon, 06 May 2024 17:32:40 GMT',
                                      'x-amzn-requestid': '43413cb5-c774-4cb0-a4a3-d32b44edf255'},
                      'HTTPStatusCode': 200,
                      'RequestId': '43413cb5-c774-4cb0-a4a3-d32b44edf255',
                      'RetryAttempts': 0},
 'creationDate': datetime.datetime(2024, 5, 6, 17, 32, 40, 11000, tzinfo=tzlocal()),
 'definition': '{"Comment": "A description of my state machine", "StartAt": '
               '"GetStartingFeats2", "States": {"GetStartingFeats2": {"Type": '
               '"Task", "Resource": "arn:aws:states:::lambda:invoke", '
               '"Parameters": {"F

### Execute step function workflow

In [13]:
# dict_response = cls_client_sfn.start_execution(
#     stateMachineArn=str_state_machine_arn,
# )
# pprint(dict_response)

### Clean-up

In [14]:
os.remove('./definition.json')